In [ ]:
import pandas as pd
from pathlib import Path
import zipfile
import re

# ==== 🔧 EDIT THESE PATHS ====

date_str = "20250606"  # Date string for the report


mapping_excel = Path(f"D:/D&T Project/Planner Report/{date_str}/mapping_file_{date_str}.xlsx")
excel_dir = Path(f"D:/D&T Project/Planner Report/{date_str}/")  # folder with site Excel files
cluster_dir = Path("D:/D&T Project/Planner Report/PAT Report Approved")  # folder with RF Cluster folders
output_dir = excel_dir  # Change if you want a separate output location

# ==== 🔁 Load Mapping File ====
df = pd.read_excel(mapping_excel)

# ==== 🔎 Regex to match site code ====
def extract_site_code(text):
    match = re.search(r"[A-Z]{3,4}\d{3,4}", str(text))
    return match.group(0) if match else None

# ==== 📁 Scan Excel files and build a lookup dict ====
excel_files = list(excel_dir.glob("*.xls*"))
excel_lookup = {
    extract_site_code(f.stem): f
    for f in excel_files
    if extract_site_code(f.stem)
}

total_excel_files = len(excel_lookup)
zipped_sites_count = 0


# ==== 📦 Process each row in mapping ====
for _, row in df.iterrows():
    site_code = str(row["New Site Code"]).strip()
    cluster_name = str(row["RF Cluster Name"]).strip()
    zip_name = str(row["File Name"]).strip() + ".zip"
    
    # === Primary Match: New Site Code ===
    excel_file = excel_lookup.get(site_code)

    # === Fallback: DU ID ===
    if not excel_file:
        du_id = str(row["DU ID"])
        fallback_code = extract_site_code(du_id)
        if fallback_code:
            excel_file = excel_lookup.get(fallback_code)
            if excel_file:
                print(f"🔁 Fallback matched DU ID: {fallback_code} → used for {site_code}")
    
    cluster_folder = cluster_dir / cluster_name
    word_files = list(cluster_folder.glob("*.docx")) if cluster_folder.exists() else []

    # ==== ✅ Check existence ====
    if not excel_file:
        print(f"❌ Excel file not found for: {site_code}")
        continue
    if not word_files:
        print(f"❌ Word file not found in cluster folder: {cluster_folder}")
        continue

    word_file = word_files[0]  # Use first word file if multiple exist

    # ==== 📦 Create ZIP ====
    zip_path = output_dir / zip_name
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        zipf.write(excel_file, arcname=excel_file.name)
        zipf.write(word_file, arcname=word_file.name)
        
    zipped_sites_count += 1
    print(f"✅ Created ZIP: {zip_path.name}")

# ==== 📊 Summary ====
print("\n📊 Summary:")
print(f"📁 Total Excel files found: {total_excel_files}")
print(f"📦 Total sites zipped: {zipped_sites_count}")
print(f"❓ Unmatched sites (not zipped): {len(df) - zipped_sites_count}")
